# Shared US trading calendar
Prices are loaded through `00_engine.load()` before percentage changes. Non-US sessions are excluded before computing returns, preserving valid returns immediately after US holidays. Fee rates and evaluation periods also come from the shared engine.


**Submission reference:** Quote historical final-portfolio performance and comparisons from the `Submission reference: historical IS / OOS metrics` table in `results/Report.html` (`results/tables/final_metrics.csv`, generated by 07). Identify the strategy row and period. This notebook's standalone performance tables are research diagnostics; their figures must not be mixed with the final account metrics.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Project root
ROOT = Path.cwd().resolve()
while not (ROOT / "pipeline" / "00_engine.ipynb").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent   # From pipeline can locate the project root from any subdirectory

# One shared calendar, fee table, and period definitions for both risk stages.
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from portfolio_accounting import ENGINE
IS_START = ENGINE['IS_START']


# Load the teams original weights
weights_path = ROOT / "results/signals/ensemble_weights_history.csv"

raw_weights = pd.read_csv(
    weights_path,
    index_col=0,
    parse_dates=True
)

# Extract only the traded assets6OnlyETF
ETFS = ["TQQQ", "UVXY", "TECL", "SPXL", "TECS", "BSV"]

raw_weights = raw_weights[ETFS]

# Check data
print("Data shape: ", raw_weights.shape)
print("Start date: ", raw_weights.index.min())
print("End date: ", raw_weights.index.max())
print("Contains missing values: ", raw_weights.isna().any().any())
print("Daily weights sum to1: ",
      np.allclose(raw_weights.sum(axis=1), 1.0))

display(raw_weights.head())

In [ ]:
# Step 2: LoadETFHistorical price data

price_path = ROOT / "data/raw/etf_daily.csv"

prices_raw = pd.read_csv(price_path)

print("Data dimensions: ", prices_raw.shape)

print("Before5rows: ")
display(prices_raw.head())

print("All column names: ")
print(prices_raw.columns.tolist())

In [ ]:
# Step 3: Correctly read the multi-level headerETFData

prices = ENGINE['load'](price_path)  # Filter US sessions BEFORE returns.

# Extract adjusted closing prices
adj_close = prices["Adj Close"][ETFS].copy()

# Sort by date
adj_close = adj_close.sort_index()

# Ensure prices are numeric
adj_close = adj_close.apply(
    pd.to_numeric,
    errors="coerce"
)

# Calculate daily simple returns
daily_returns = adj_close.pct_change(fill_method=None)

# Check data
print("Price data shape: ", adj_close.shape)

print("Start date: ", adj_close.index.min())
print("End date: ", adj_close.index.max())

print("\nEachETFNumber of missing values: ")
print(adj_close.isna().sum())

print("\nFirst rows of daily returns5rows: ")
display(daily_returns.head())

In [ ]:
# Step 4.1: Check weight dates for completeETFPrice

# Align with the original team weight dates
aligned_prices = adj_close.reindex(raw_weights.index)

# Count missing values
print("Missing prices after alignment to weight dates: ")
print(aligned_prices.isna().sum())

# Find dates with missing values
missing_dates = aligned_prices[
    aligned_prices.isna().any(axis=1)
]

print("\nNumber of dates with missing prices: ", len(missing_dates))

display(missing_dates.head(10))

# Check the last few days of data
print("\nMost recent5days of prices: ")
display(aligned_prices.tail())

In [ ]:
# Step 4.2: Calculate daily forecast annualized volatility of the original portfolio

LOOKBACK = 60
TRADING_DAYS = 252

# Ensure return dates are in ascending order
daily_returns = daily_returns.sort_index()

# Use complete observations only6OnlyETFReturns
valid_returns = daily_returns[ETFS].dropna()

# Check2012year8month1Sufficient historical data before this date
first_date = raw_weights.index.min()

history_before_start = valid_returns.loc[
    valid_returns.index < first_date
]

print("Complete historical return days before strategy start: ",
      len(history_before_start))

assert len(history_before_start) >= LOOKBACK, (
    "Insufficient data before strategy start60complete trading days, Check price data."
)

portfolio_vol = pd.Series(
    index=raw_weights.index,
    dtype=float,
    name="Predicted_Volatility"
)

for date in raw_weights.index:

    # 1. Take the most recent observations through that day60trading days of returns
    past_returns = valid_returns.loc[:date].tail(LOOKBACK)

    if len(past_returns) < LOOKBACK:
        continue

    # 2. Same day6OnlyETForiginal weights
    w = raw_weights.loc[date, ETFS].to_numpy(dtype=float)

    # 3. Calculate60day covariance matrix
    cov_matrix = past_returns.cov().to_numpy()

    # 4. Calculate annualized portfolio volatility
    variance = w @ cov_matrix @ w

    portfolio_vol.loc[date] = np.sqrt(
        TRADING_DAYS * max(variance, 0)
    )

# View results
print("\nForecast annualized volatility statistics: ")
display(portfolio_vol.describe())

print("\nBefore5days of forecast annualized volatility: ")
display(portfolio_vol.head())

print("\nMost recent5days of forecast annualized volatility: ")
display(portfolio_vol.tail())

print("\nDays with unavailable volatility estimates: ",
      portfolio_vol.isna().sum())

In [ ]:
# Step 5: Based on eachETFAdjust original weights using historical volatility

LOOKBACK = 60
TRADING_DAYS = 252

adjusted_weights = pd.DataFrame(
    index=raw_weights.index,
    columns=ETFS,
    dtype=float
)

for date in raw_weights.index:

    # 1. Get the teams original weights for the day
    w_raw = raw_weights.loc[date, ETFS]

    # 2. Get recent observations through the current day60trading days of returns
    past_returns = valid_returns.loc[:date].tail(LOOKBACK)

    # 3. Calculate each assetsETFannualized volatility
    asset_vol = past_returns.std() * np.sqrt(TRADING_DAYS)

    # 4. Compute scores using original weights and volatility
    scores = w_raw / asset_vol

    # 5. Standardized, Make all weights sum to100%
    w_final = scores / scores.sum()

    # 6. Save final weights
    adjusted_weights.loc[date] = w_final

# Check results
print("Weights sum on all dates to100%: ")
print(np.allclose(adjusted_weights.sum(axis=1), 1.0))

print("\nContains negative weights: ")
print((adjusted_weights < 0).any().any())

print("\nMost recent5days of original weights: ")
display(raw_weights.tail())

print("\nMost recent5days of adjusted weights: ")
display(adjusted_weights.tail())

print("\nAverage weight comparison: ")
comparison = pd.DataFrame({
    "Original": raw_weights.mean(),
    "Adjusted": adjusted_weights.mean()
})

display(comparison)

In [ ]:
# Step 6: Compare forecast annualized volatility before and after risk adjustment

adjusted_portfolio_vol = pd.Series(
    index=raw_weights.index,
    dtype=float,
    name="Adjusted_Volatility"
)

for date in raw_weights.index:

    # Get historical60days of historical returns
    past_returns = valid_returns.loc[:date].tail(LOOKBACK)

    # Calculate covariance matrix
    cov_matrix = past_returns.cov().to_numpy()

    # Get adjustedETFWeights
    w = adjusted_weights.loc[date, ETFS].to_numpy(dtype=float)

    # Calculate adjusted portfolio volatility
    variance = w @ cov_matrix @ w

    adjusted_portfolio_vol.loc[date] = np.sqrt(
        TRADING_DAYS * max(variance, 0)
    )

# Compare original and adjusted portfolios
vol_comparison = pd.DataFrame({
    "Original_Vol": portfolio_vol,
    "Adjusted_Vol": adjusted_portfolio_vol
})

vol_comparison["Vol_Change"] = (
    vol_comparison["Adjusted_Vol"]
    - vol_comparison["Original_Vol"]
)

print("Original and adjusted forecast volatility statistics: ")
display(vol_comparison.describe())

print("\nProportion of trading days with lower risk: ")
print(
    f"{(vol_comparison['Vol_Change'] < -1e-8).mean():.2%}"
)

print("\nProportion of trading days with higher risk: ")
print(
    f"{(vol_comparison['Vol_Change'] > 1e-8).mean():.2%}"
)

print("\nMost recent5days of risk comparisons: ")
display(vol_comparison.tail())

In [ ]:
# ============================================================
# Step 7: Use the original team backtest to compute both strategy equity curves
# ============================================================

import ast
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Load the teamsNotebook
# ------------------------------------------------------------

notebook_path = next((ROOT / "pipeline").rglob("02_parameter_ensemble.ipynb"))

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)


# ------------------------------------------------------------
# 2. Define parameters used by the original team backtest
# ------------------------------------------------------------

IS_START = ENGINE['IS_START']

TICKS = ETFS

# Original team one-way transaction costs (Unit: bps)
ONE_WAY_BPS = {t: ENGINE['ONE_WAY_BPS'][t] for t in ETFS}


# ------------------------------------------------------------
# 3. Extract the teams backtest function
# ------------------------------------------------------------

# Shared exact transaction-cost solver used by the extracted portfolio function.
_engine_ns = ENGINE
rebalance_exact = ENGINE['rebalance_exact']

def load_function(notebook, function_name):

    for cell in notebook["cells"]:

        if cell["cell_type"] != "code":
            continue

        source = "".join(cell["source"])

        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue

        for node in tree.body:

            if (
                isinstance(node, ast.FunctionDef)
                and node.name == function_name
            ):

                module = ast.Module(
                    body=[node],
                    type_ignores=[]
                )

                exec(
                    compile(module, "<notebook>", "exec"),
                    globals()
                )

                return

    raise ValueError(f"Cannot find function: {function_name}")


load_function(nb, "backtest_weighted_portfolio")

print("Backtest function loaded successfully!")


# ------------------------------------------------------------
# 4. PrepareETFOpening and closing prices
# ------------------------------------------------------------

price_data = prices.sort_index()

# Unadjusted closing prices
close_unadjusted = (
    price_data["Close"][ETFS]
    .astype(float)
)

# Adjusted closing prices
close_adjusted = (
    price_data["Adj Close"][ETFS]
    .astype(float)
)

# Calculate price adjustment factors
adjustment_factor = (
    close_adjusted / close_unadjusted
)

# Adjust opening prices to the same basis
open_adjusted = (
    price_data["Open"][ETFS].astype(float)
    * adjustment_factor
)


# ------------------------------------------------------------
# 5. Align trading dates
# ------------------------------------------------------------

dates = raw_weights.index

O = open_adjusted.reindex(dates)
C = close_adjusted.reindex(dates)

assert O.notna().all().all(), "Missing opening prices"
assert C.notna().all().all(), "Missing closing prices"

assert raw_weights.index.equals(adjusted_weights.index), (
    "Original and adjusted weight dates do not match"
)

assert np.allclose(
    raw_weights.sum(axis=1), 1.0
), "Original weights do not sum to100%"

assert np.allclose(
    adjusted_weights.sum(axis=1), 1.0
), "Adjusted weights do not sum to100%"


# ------------------------------------------------------------
# 6. Backtest original ensemble strategy
# ------------------------------------------------------------

original_equity, original_diag = backtest_weighted_portfolio(
    O,
    C,
    raw_weights,
    start=IS_START
)

print("Original strategy backtest complete!")


# ------------------------------------------------------------
# 7. Backtest the risk-adjusted strategy
# ------------------------------------------------------------

adjusted_equity, adjusted_diag = backtest_weighted_portfolio(
    O,
    C,
    adjusted_weights,
    start=IS_START
)

print("Risk-adjusted strategy backtest complete!")


# ------------------------------------------------------------
# 8. Check results
# ------------------------------------------------------------

equity_comparison = pd.DataFrame({
    "Original": original_equity,
    "Risk_Adjusted": adjusted_equity
})

print("\n========== Most recent5days of account equity ==========")
display(equity_comparison.tail())

print("\n========== Final account equity ==========")

print(f"Original strategy: {original_equity.iloc[-1]:.4f}")
print(f"Risk adjustment: {adjusted_equity.iloc[-1]:.4f}")

print("\n========== Data checks ==========")

print("Number of trading days: ", len(equity_comparison))

print(
    "Missing values in either equity series: ",
    equity_comparison.isna().any().any()
)

print(
    "Both date indexes match: ",
    original_equity.index.equals(adjusted_equity.index)
)


In [ ]:
# ============================================================
# Step 8: Use the original teammetricsFunction computes strategy performance
# ============================================================

import ast
import json
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Load the teams core engine
# ------------------------------------------------------------

engine_path = ROOT / "pipeline/00_engine.ipynb"

with open(engine_path, "r", encoding="utf-8") as f:
    engine_nb = json.load(f)


# ------------------------------------------------------------
# 2. Extract the original team performance function
# ------------------------------------------------------------

def load_metrics_function(notebook):

    for cell in notebook["cells"]:

        if cell["cell_type"] != "code":
            continue

        source = "".join(cell["source"])

        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue

        for node in tree.body:

            if (
                isinstance(node, ast.FunctionDef)
                and node.name == "metrics"
            ):

                module = ast.Module(
                    body=[node],
                    type_ignores=[]
                )

                exec(
                    compile(module, "<engine>", "exec"),
                    globals()
                )

                return

    raise ValueError("Cannot findmetricsFunction")


load_metrics_function(engine_nb)

print("Original team performance function loaded successfully!")


# ------------------------------------------------------------
# 3. Define test periods
# ------------------------------------------------------------

periods = {
    "IS": (ENGINE["IS_START"], ENGINE["IS_END"]),
    "OOS": (ENGINE["OOS_START"], ENGINE["OOS_END"]),
    "OOS ex-2020": ("2021-01-01", ENGINE["OOS_END"])
}


# ------------------------------------------------------------
# 4. Calculate performance of both strategies
# ------------------------------------------------------------

rows = []

for period, (start, end) in periods.items():

    for name, equity in [
        ("Original Ensemble", original_equity),
        ("Risk Adjusted", adjusted_equity)
    ]:

        result = metrics(
            equity,
            a=start,
            b=end
        )

        rows.append({
            "Strategy": name,
            "Period": period,
            **result
        })


# ------------------------------------------------------------
# 5. Summarize results
# ------------------------------------------------------------

performance_comparison = pd.DataFrame(rows)

print("\n========== Performance before and after risk adjustment ==========")

display(performance_comparison.round(4))


# ------------------------------------------------------------
# 6. View out-of-sample results separately
# ------------------------------------------------------------

oos_performance = performance_comparison[
    performance_comparison["Period"] == "OOS"
]

print("\n========== Out-of-sample performance ==========")

display(oos_performance.round(4))


# ------------------------------------------------------------
# 7. Check against previously saved team results
# ------------------------------------------------------------

saved_metrics_path = (
    ROOT / "results/tables/ensemble_metrics.csv"
)

saved_metrics = pd.read_csv(saved_metrics_path)

saved_ensemble = saved_metrics[
    saved_metrics["Strategy"] == "Ensemble 6336"
]

print("\n========== Original strategy performance saved by the team ==========")

display(saved_ensemble.round(4))

In [ ]:
# ============================================================
# Step 9: Different risk adjustment strengths Alpha robustness tests
# ============================================================

ALPHAS = [0, 0.5, 1, 1.5]

LOOKBACK = 60

alpha_results = []
alpha_weights = {}
alpha_equities = {}

# Precompute daily values for eachETFhistorical annualized volatility
asset_vol_history = pd.DataFrame(
    index=raw_weights.index,
    columns=ETFS,
    dtype=float
)

for date in raw_weights.index:

    past_returns = valid_returns.loc[:date].tail(LOOKBACK)

    asset_vol_history.loc[date] = (
        past_returns.std() * np.sqrt(252)
    ).reindex(ETFS).values

# Check historical volatility
assert asset_vol_history.notna().all().all()
assert (asset_vol_history > 0).all().all()

# Test different values separatelyAlpha
for alpha in ALPHAS:

    # 1. Based onAlphaCalculate risk adjustment scores
    scores = raw_weights.div(
        asset_vol_history ** alpha
    )

    # 2. Normalize weights
    weights = scores.div(
        scores.sum(axis=1),
        axis=0
    )

    # 3. Check weights
    assert np.allclose(weights.sum(axis=1), 1.0)
    assert (weights >= 0).all().all()

    # 4. Save weights
    alpha_weights[alpha] = weights

    # 5. Use the teams backtest function
    equity, diag = backtest_weighted_portfolio(
        O,
        C,
        weights,
        start=IS_START
    )

    alpha_equities[alpha] = equity

    # 6. Calculate performance for the three periods separately
    for period, (start, end) in periods.items():

        result = metrics(
            equity,
            a=start,
            b=end
        )

        alpha_results.append({
            "Alpha": alpha,
            "Period": period,
            **result
        })

# 7. Organize results
alpha_comparison = pd.DataFrame(alpha_results)

print("========== DifferentAlphaperformance ==========")

display(alpha_comparison.round(4))


# 8. View in-sample performance separately
print("\n========== In-sample IS ==========")

display(
    alpha_comparison[
        alpha_comparison["Period"] == "IS"
    ].round(4)
)


# 9. View out-of-sample performance separately
print("\n========== Out-of-sample OOS ==========")

display(
    alpha_comparison[
        alpha_comparison["Period"] == "OOS"
    ].round(4)
)


# 10. View excluding2020year results
print("\n========== OOS ex-2020 ==========")

display(
    alpha_comparison[
        alpha_comparison["Period"] == "OOS ex-2020"
    ].round(4)
)

In [ ]:
# ============================================================
# Step 10.1: Save risk-adjusted strategy research results
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Create output directories
# ------------------------------------------------------------

TABLE_DIR = ROOT / "results" / "tables"
SIGNAL_DIR = ROOT / "results" / "signals"
FIGURE_DIR = ROOT / "results" / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
SIGNAL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. Confirm use of Alpha = 1
# ------------------------------------------------------------

SELECTED_ALPHA = 1.0

final_weights = alpha_weights[SELECTED_ALPHA].copy()
final_equity = alpha_equities[SELECTED_ALPHA].copy()

# Check weights
assert final_weights.index.equals(raw_weights.index)

assert np.allclose(
    final_weights.sum(axis=1),
    1.0
), "Some weight sums differ from100%dates"

assert (final_weights >= 0).all().all(), "Negative weights exist"

assert final_weights.notna().all().all(), "Missing weights exist"


# ------------------------------------------------------------
# 3. Save daily risk-adjusted weights
# ------------------------------------------------------------

weights_output = (
    SIGNAL_DIR / "risk_adjusted_weights.csv"
)

final_weights.to_csv(
    weights_output,
    index_label="Date"
)

print("Daily adjusted weights saved: ")
print(weights_output)


# ------------------------------------------------------------
# 4. Save the main performance comparison
# ------------------------------------------------------------

main_performance = performance_comparison.copy()

performance_output = (
    TABLE_DIR / "risk_performance.csv"
)

main_performance.to_csv(
    performance_output,
    index=False
)

print("\nMain strategy performance saved: ")
print(performance_output)


# ------------------------------------------------------------
# 5. Save Alpha Robustness tests
# ------------------------------------------------------------

robustness_output = (
    TABLE_DIR / "risk_alpha_robustness.csv"
)

alpha_comparison.to_csv(
    robustness_output,
    index=False
)

print("\nSavedAlphaRobustness tests: ")
print(robustness_output)


# ------------------------------------------------------------
# 6. Save daily equity for the original and risk-adjusted strategies
# ------------------------------------------------------------

equity_output = pd.DataFrame({
    "Original": original_equity,
    "Risk_Adjusted": final_equity
})

equity_file = (
    TABLE_DIR / "risk_equity_history.csv"
)

equity_output.to_csv(
    equity_file,
    index_label="Date"
)

print("\nDaily equity saved: ")
print(equity_file)


# ------------------------------------------------------------
# 7. Display output files and sizes
# ------------------------------------------------------------

print("\n========== Output file checks ==========")

files = [
    weights_output,
    performance_output,
    robustness_output,
    equity_file
]

for file in files:

    print(
        f"{file.name}: "
        f"{file.stat().st_size / 1024:.2f} KB"
    )


# ------------------------------------------------------------
# 8. Display latest adjusted weights
# ------------------------------------------------------------

print("\n========== Latest risk-adjusted weights ==========")

display(
    final_weights.tail(1).T.rename(
        columns={
            final_weights.index[-1]: "Weight"
        }
    )
)

In [ ]:
# ============================================================
# Step 10.2: Original versus risk-adjusted cumulative equity
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

# Original ensemble strategy
plt.plot(
    original_equity.index,
    original_equity.values,
    label="Original Ensemble",
    linewidth=2
)

# Risk-adjusted strategy
plt.plot(
    final_equity.index,
    final_equity.values,
    label="Risk Adjusted (Alpha = 1)",
    linewidth=2
)

plt.title(
    "Cumulative Equity: Original vs Risk Adjusted",
    fontsize=14
)

plt.xlabel("Date")
plt.ylabel("Portfolio Equity")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "risk_equity_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# Step 10.3: Maximum drawdown comparison
# ============================================================

# Calculate daily drawdown
original_drawdown = (
    original_equity / original_equity.cummax() - 1
)

adjusted_drawdown = (
    final_equity / final_equity.cummax() - 1
)

plt.figure(figsize=(12, 6))

plt.plot(
    original_drawdown.index,
    original_drawdown.values * 100,
    label="Original Ensemble",
    linewidth=1.5
)

plt.plot(
    adjusted_drawdown.index,
    adjusted_drawdown.values * 100,
    label="Risk Adjusted (Alpha = 1)",
    linewidth=1.5
)

plt.title(
    "Portfolio Drawdown Comparison",
    fontsize=14
)

plt.xlabel("Date")
plt.ylabel("Drawdown (%)")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "risk_drawdown_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Full-period maximum drawdown:")

print(
    f"Original: {original_drawdown.min():.2%}"
)

print(
    f"Risk Adjusted: {adjusted_drawdown.min():.2%}"
)

In [ ]:
# ============================================================
# Step 10.4: TQQQAllocation adjustment comparison
# ============================================================

plt.figure(figsize=(12, 5))

plt.plot(
    raw_weights.index,
    raw_weights["TQQQ"] * 100,
    label="Original TQQQ Weight",
    alpha=0.7
)

plt.plot(
    final_weights.index,
    final_weights["TQQQ"] * 100,
    label="Risk Adjusted TQQQ Weight",
    alpha=0.7
)

plt.title(
    "TQQQ Portfolio Weight Comparison",
    fontsize=14
)

plt.xlabel("Date")
plt.ylabel("Portfolio Weight (%)")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "risk_weight_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# Step 11: Volatility lookback robustness tests
# ============================================================

LOOKBACKS = [30, 60, 90, 120]
ALPHA = 1.0
TRADING_DAYS = 252

lookback_results = []
lookback_weights = {}
lookback_equities = {}

# ------------------------------------------------------------
# 1. Test different historical lookbacks
# ------------------------------------------------------------

for lookback in LOOKBACKS:

    print(f"Testing Lookback = {lookback} ...")

    # Create a daily volatility data frame
    vol_history = pd.DataFrame(
        index=raw_weights.index,
        columns=ETFS,
        dtype=float
    )

    # --------------------------------------------------------
    # 2. Calculate each daysETFhistorical volatility
    # --------------------------------------------------------

    for date in raw_weights.index:

        past_returns = (
            valid_returns
            .loc[:date, ETFS]
            .tail(lookback)
        )

        # Ensure sufficient historical data
        if len(past_returns) < lookback:
            continue

        vol_history.loc[date] = (
            past_returns.std()
            * np.sqrt(TRADING_DAYS)
        ).values

    # --------------------------------------------------------
    # 3. Check volatility data
    # --------------------------------------------------------

    assert vol_history.notna().all().all(), (
        f"Lookback={lookback} Missing volatility values exist"
    )

    assert (vol_history > 0).all().all(), (
        f"Lookback={lookback} Invalid volatility values exist"
    )

    # --------------------------------------------------------
    # 4. Recalculate weights using volatility
    # --------------------------------------------------------

    scores = raw_weights.div(
        vol_history ** ALPHA
    )

    weights = scores.div(
        scores.sum(axis=1),
        axis=0
    )

    # Ensure daily weights sum to100%
    assert np.allclose(
        weights.sum(axis=1),
        1.0
    )

    # Ensure no negative weights
    assert (weights >= 0).all().all()

    # Save daily weights
    lookback_weights[lookback] = weights

    # --------------------------------------------------------
    # 5. Use the team backtest function
    # --------------------------------------------------------

    equity, diag = backtest_weighted_portfolio(
        O,
        C,
        weights,
        start=IS_START
    )

    lookback_equities[lookback] = equity

    # --------------------------------------------------------
    # 6. Calculate performance for each period
    # --------------------------------------------------------

    for period, (start, end) in periods.items():

        result = metrics(
            equity,
            a=start,
            b=end
        )

        lookback_results.append({
            "Lookback": lookback,
            "Alpha": ALPHA,
            "Period": period,
            **result
        })

    print(f"Lookback = {lookback} Complete!")


# ------------------------------------------------------------
# 7. Summarize results
# ------------------------------------------------------------

lookback_comparison = pd.DataFrame(
    lookback_results
)

print("\n========== All lookback test results ==========")

display(
    lookback_comparison.round(4)
)


# ------------------------------------------------------------
# 8. In-sample results
# ------------------------------------------------------------

print("\n========== IS In-sample ==========")

display(
    lookback_comparison[
        lookback_comparison["Period"] == "IS"
    ].round(4)
)


# ------------------------------------------------------------
# 9. Out-of-sample results
# ------------------------------------------------------------

print("\n========== OOS Out-of-sample ==========")

display(
    lookback_comparison[
        lookback_comparison["Period"] == "OOS"
    ].round(4)
)


# ------------------------------------------------------------
# 10. Excludes2020year
# ------------------------------------------------------------

print("\n========== OOS ex-2020 ==========")

display(
    lookback_comparison[
        lookback_comparison["Period"] == "OOS ex-2020"
    ].round(4)
)


# ------------------------------------------------------------
# 11. Check60day lookback reproduces original results
# ------------------------------------------------------------

print("\n========== 60day lookback consistency check ==========")

same_weights = np.allclose(
    lookback_weights[60][ETFS].values,
    adjusted_weights[ETFS].values
)

print("andStep 5weights match: ", same_weights)

assert same_weights, "60day lookback failed to reproduce previous weights!"

In [ ]:
# Save volatility lookback robustness results

lookback_output = (
    ROOT / "results/tables/risk_lookback_robustness.csv"
)

lookback_comparison.to_csv(
    lookback_output,
    index=False
)

print("Volatility lookback robustness results saved: ")
print(lookback_output)